# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:  
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

The `mlcroissant` library allows us to inspect the dataset's main entities by their `@id`. Here, we look for available record sets, and within them, observe their fields and columns by `@id` as well.

Let's print all available record sets:

In [ ]:
# List all available record sets by @id and name
record_sets = dataset.record_sets

if not record_sets:
    print("No record sets found in this dataset.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs.id}, name: {rs.name}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field @id: {field.id}, name: {field.name}")
        if hasattr(rs, 'columns') and rs.columns:
            print("  Columns:")
            for col in rs.columns:
                print(f"    - Column @id: {col.id}, name: {col.name}")

If record sets are available, we can explore them further. For demonstration, we'll attempt to load records from each record set (printing the first 2 records):

In [ ]:
for rs in record_sets:
    print(f"\n=== Records for RecordSet @id: {rs.id}, name: {rs.name} ===")
    for idx, record in enumerate(dataset.records(record_set=rs.id)):
        print(record)
        if idx>=1:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All record sets and fields should be referenced by their `@id`.

We'll build a dictionary of DataFrames, keyed by each record set's `@id`.

In [ ]:
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

if dataframes:
    first_rs_id = record_set_ids[0]
    print(f"Fields (columns) in RecordSet @id {first_rs_id}:")
    print(dataframes[first_rs_id].columns.tolist())
    display(dataframes[first_rs_id].head())
else:
    print("No dataframes were extracted - check that record sets are defined and data is available.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping or aggregating data. All field references use their `@id`.

> **Note:** If the dataset or record sets contain no rows or fields, you may need to adapt the code below to your specific schema after running previous cells.

In [ ]:
# Choose record set and fields for numeric analysis
if dataframes:
    # Try to find the first dataframe with at least one numeric column
    chosen_df = None
    chosen_rs_id = None
    numeric_field_id = None
    group_field_id = None
    for rs_id, df in dataframes.items():
        numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
        if numeric_cols:
            chosen_df = df
            chosen_rs_id = rs_id
            numeric_field_id = numeric_cols[0]
            # Try to find a candidate group field
            obj_or_str_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
            if obj_or_str_cols:
                group_field_id = obj_or_str_cols[0]
            break

    if numeric_field_id is not None:
        print(f"Using RecordSet @id: {chosen_rs_id}")
        print(f"Numeric Field @id: {numeric_field_id}")
        if group_field_id:
            print(f"Group Field @id: {group_field_id}")
        else:
            print("No suitable group field found. Proceeding without grouping.")

        threshold = chosen_df[numeric_field_id].mean() if chosen_df[numeric_field_id].notnull().any() else 0
        filtered_df = chosen_df[chosen_df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id, as_index=False)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
    else:
        print("No numeric columns found for EDA.")
else:
    print("No dataframes to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. This could include histograms, boxplots, or scatterplots for numeric fields, or bar plots/count plots for categorical fields (using their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

if dataframes and numeric_field_id is not None and chosen_df is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(chosen_df[numeric_field_id].dropna(), kde=True, bins=30, color='deepskyblue')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in chosen_df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=chosen_df, x=group_field_id, y=numeric_field_id)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- We loaded the FAIR^2 ordered logistic regression dataset using its Croissant schema, examined available record sets, and explored their fields using their `@id` identifiers.
- DataFrames were constructed (if possible) for each record set, and exploratory analysis was performed on available numeric fields: filtering, normalization, and simple grouping.
- Visualizations illustrated the distribution and spread of numeric data and its groupwise behavior where possible.
- Due to the FAIR-first approach, all programmatic access (including filtering or grouping) uses `@id` for referencing fields and record sets, ensuring reproducibility and interoperability.

You can now extend this notebook for deeper domain-specific analyses or integrate with additional ML and statistics libraries as needed.